<a href="https://colab.research.google.com/github/shoh0806/Deep-Learning-Project/blob/main/Cross_Attention_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. 라이브러리 설치

In [ ]:
!pip install transformers accelerate -q

# 2. 라이브러리 import, 시드 고정(Bert, Vit와 통일)

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import BertTokenizer, BertModel
from transformers import ViTModel, ViTImageProcessor

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")

def set_seed(seed=42):                        # 승현님이 설정하신 조건과 같게 시드는 42로 설정하였습니다.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# 3. 구글 드라이브 연결 및 데이터 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/clickbait_project'
CSV_PATH = f'{BASE_DIR}/dataset.csv'
IMG_DIR  = f'{BASE_DIR}/thumbnails'

df = pd.read_csv(CSV_PATH)

print(df.head())
print(df['label'].value_counts())

# 4. 데이터셋 split

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(len(train_df), len(val_df), len(test_df))

# 승현님이 말씀해주신대로 random_state를 42로 설정하였습니다. 또한 데이터셋 split도 동일하게 진행하였습니다.(공정성 유지)

# 5. Multimodal Dataset

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
image_processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')

class MultimodalDataset(Dataset):
    def __init__(self, df, tokenizer, image_processor, img_dir, max_len=64):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.image_processor = image_processor
        self.img_dir = img_dir
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        title = str(row['title'])
        label = int(row['label'])
        video_id = row['video_id']

        text_inputs = self.tokenizer(
            title,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        img_path = f"{self.img_dir}/{video_id}.jpg"
        image = Image.open(img_path).convert("RGB")

        image_inputs = self.image_processor(
            images=image,
            return_tensors='pt'
        )

        return {
            'input_ids': text_inputs['input_ids'].squeeze(0),
            'attention_mask': text_inputs['attention_mask'].squeeze(0),
            'pixel_values': image_inputs['pixel_values'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }

# 6. DataLoader

In [ ]:
BATCH_SIZE = 16        # 배치 사이즈를 8로 한 이유는 GPU 메모리 때문입니다!
MAX_LEN = 128

train_dataset = MultimodalDataset(train_df, tokenizer, image_processor, IMG_DIR, MAX_LEN)
val_dataset   = MultimodalDataset(val_df, tokenizer, image_processor, IMG_DIR, MAX_LEN)
test_dataset  = MultimodalDataset(test_df, tokenizer, image_processor, IMG_DIR, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Dataset 준비 완료!")
print(f"train loader: {len(train_loader)}배치")

# 7. Cross-Attention 모델(bert, vit 마지막 2개 layer만 오픈)

In [ ]:
def freeze_backbone_except_last_layers(model, bert_unfreeze_layers=2, vit_unfreeze_blocks=2):
    """
    BERT와 ViT backbone은 대부분 freeze하고,
    마지막 일부 layer/block만 fine-tuning하도록 설정하는 함수
    """

    # ==============================
    # 1. BERT 전체 freeze
    # ==============================
    for param in model.bert.parameters():
        param.requires_grad = False

    # BERT 마지막 N개 encoder layer만 unfreeze
    for layer in model.bert.encoder.layer[-bert_unfreeze_layers:]:
        for param in layer.parameters():
            param.requires_grad = True

    # ==============================
    # 2. ViT 전체 freeze
    # ==============================
    for param in model.vit.parameters():
        param.requires_grad = False

    # ViT 마지막 N개 encoder block만 unfreeze
    for layer in model.vit.encoder.layer[-vit_unfreeze_blocks:]:
        for param in layer.parameters():
            param.requires_grad = True

    # ViT는 encoder 뒤에 layernorm이 있으므로 같이 열어주는 것이 좋음
    if hasattr(model.vit, "layernorm"):
        for param in model.vit.layernorm.parameters():
            param.requires_grad = True

In [ ]:
class CrossAttentionClickbaitModel(nn.Module):
    def __init__(
        self,
        bert_name='bert-base-multilingual-cased',
        vit_name='google/vit-base-patch16-224',
        hidden_size=768,
        num_heads=8,
        dropout=0.3,
        freeze_backbone=False
    ):
        super().__init__()

        self.bert = BertModel.from_pretrained(bert_name)
        self.vit = ViTModel.from_pretrained(vit_name)

        self.text_to_image_attn = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.image_to_text_attn = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm_t2i = nn.LayerNorm(hidden_size)
        self.norm_i2t = nn.LayerNorm(hidden_size)

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 4, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 2)
        )

        # BERT와 ViT는 마지막 2개 layer/block만 fine-tuning
        freeze_backbone_except_last_layers(
            self,
            bert_unfreeze_layers=2,
            vit_unfreeze_blocks=2
        )

    def forward(self, input_ids, attention_mask, pixel_values):
        bert_outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        vit_outputs = self.vit(
            pixel_values=pixel_values
        )

        text_hidden = bert_outputs.last_hidden_state
        image_hidden = vit_outputs.last_hidden_state

        text_cls = text_hidden[:, 0, :]
        image_cls = image_hidden[:, 0, :]

        text_padding_mask = attention_mask == 0

        t2i_output, _ = self.text_to_image_attn(
            query=text_hidden,
            key=image_hidden,
            value=image_hidden
        )

        i2t_output, _ = self.image_to_text_attn(
            query=image_hidden,
            key=text_hidden,
            value=text_hidden,
            key_padding_mask=text_padding_mask
        )

        t2i_output = self.norm_t2i(t2i_output + text_hidden)
        i2t_output = self.norm_i2t(i2t_output + image_hidden)

        t2i_cls = t2i_output[:, 0, :]
        i2t_cls = i2t_output[:, 0, :]

        fused = torch.cat(
            [text_cls, image_cls, t2i_cls, i2t_cls],
            dim=1
        )

        fused = self.dropout(fused)
        logits = self.classifier(fused)

        return logits

# 8. 기존 BERT / ViT fine-tuned weight 불러오기

In [ ]:
def load_backbone_weights(model, BASE_DIR):
    bert_path = f"{BASE_DIR}/mbert_cased_best.pt"
    vit_path = f"{BASE_DIR}/vit_best.pt"

    if os.path.exists(bert_path):
        bert_state = torch.load(bert_path, map_location='cpu')
        bert_only = {
            k.replace("bert.", ""): v
            for k, v in bert_state.items()
            if k.startswith("bert.")
        }
        model.bert.load_state_dict(bert_only, strict=False)
        print("BERT fine-tuned weight loaded.")

    else:
        print("BERT weight not found. Using pretrained BERT.")

    if os.path.exists(vit_path):
        vit_state = torch.load(vit_path, map_location='cpu')
        vit_only = {
            k.replace("vit.", ""): v
            for k, v in vit_state.items()
            if k.startswith("vit.")
        }
        model.vit.load_state_dict(vit_only, strict=False)
        print("ViT fine-tuned weight loaded.")

    else:
        print("ViT weight not found. Using pretrained ViT.")

# 9. 모델 생성

In [ ]:
model = CrossAttentionClickbaitModel(
    dropout=0.3,
    num_heads=8
).to(device)

load_backbone_weights(model, BASE_DIR)

print("Cross-Attention 모델 생성 완료!")
print(f"학습가능 파라미터 수: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"전체 파라미터 수:     {sum(p.numel() for p in model.parameters()):,}")

# 10. Optimizer, train / eval 함수

In [ ]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=3e-5,
    weight_decay=0
)

criterion = nn.CrossEntropyLoss()

def train_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0
    preds = []
    targets = []

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values
        )

        loss = criterion(outputs, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()
        preds += outputs.argmax(dim=1).detach().cpu().tolist()
        targets += labels.detach().cpu().tolist()

    avg_loss = total_loss / len(loader)
    f1 = f1_score(targets, preds, average='macro')
    acc = accuracy_score(targets, preds)

    return avg_loss, f1, acc


def eval_epoch(model, loader, criterion):
    model.eval()

    total_loss = 0
    preds = []
    targets = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            pixel_values = batch['pixel_values'].to(device)
            labels = batch['label'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values
            )

            loss = criterion(outputs, labels)

            total_loss += loss.item()
            preds += outputs.argmax(dim=1).cpu().tolist()
            targets += labels.cpu().tolist()

    avg_loss = total_loss / len(loader)
    f1 = f1_score(targets, preds, average='macro')
    acc = accuracy_score(targets, preds)

    return avg_loss, f1, acc, preds, targets

# 11. 학습

In [ ]:
EPOCHS = 10
PATIENCE = 3

best_val_f1 = 0
patience_counter = 0

SAVE_PATH = f"{BASE_DIR}/cross_attention_v2_best.pt"

for epoch in range(EPOCHS):
    train_loss, train_f1, train_acc = train_epoch(
        model, train_loader, optimizer, criterion
    )

    val_loss, val_f1, val_acc, _, _ = eval_epoch(
        model, val_loader, criterion
    )

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"  train loss: {train_loss:.4f} | f1: {train_f1:.4f} | acc: {train_acc:.4f}")
    print(f"  val   loss: {val_loss:.4f} | f1: {val_f1:.4f} | acc: {val_acc:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), SAVE_PATH)
        print("  → best model saved")
    else:
        patience_counter += 1
        print(f"  → patience {patience_counter}/{PATIENCE}")

        if patience_counter >= PATIENCE:
            print("Early stopping")
            break

# 12. 테스트 평가

In [ ]:
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))

test_loss, test_f1, test_acc, test_preds, test_targets = eval_epoch(
    model, test_loader, criterion
)

print("========== Cross-Attention 최종 결과 ==========")
print(f"test loss: {test_loss:.4f}")
print(f"test F1:   {test_f1:.4f}")
print(f"test Acc:  {test_acc:.4f}")

print(classification_report(test_targets, test_preds, digits=4))
print(confusion_matrix(test_targets, test_preds))